# Aula 14 · Hands-on: automatizando cargas de trabalho com agendamento e gatilhos

## O que é
A configuração prática dos disparos automáticos no job `growdev_pipeline_emprestimos`, criado no lab da aula 12.

## Por que importa
Um job que depende de alguém clicar em **Run now** não está pronto para produção.

## Como aplicar
### Parte 1: agendamento
1. Abra o job → **Schedules & Triggers** → **Add trigger**.
2. **Trigger type:** Scheduled → **Advanced**.
3. Cron: `0 * * * * ?` · **Timezone:** `(UTC-03:00) America/Sao_Paulo`.
4. **Save**.
5. **Pausar:** no mesmo painel, clique em **Pause**. O agendamento continua salvo, mas não dispara.
6. **Retomar:** clique em **Resume**.

### Parte 2: gatilho por chegada de arquivo
Um job tem **um** tipo de gatilho por vez. Troque o agendamento pela campainha:

1. **Schedules & Triggers** → edite o trigger → **Trigger type:** File arrival.
2. **Storage location:** `/Volumes/growdev/bronze/landing/loans/`
3. **Advanced** (opcional):
   - *Minimum time between triggers:* `60` segundos (evita disparos em rajada).
   - *Wait after last change:* `60` segundos (espera o arquivo terminar de ser gravado).
4. **Save**.

### Parte 3: simulando a chegada de dados
Rode a célula abaixo. Ela entrega um arquivo novo em `landing/loans/`, como se o parceiro tivesse enviado.

In [0]:
dbutils.notebook.run("../Projeto/utilitarios/simular_chegada_lote", 300, {"catalogo": "growdev"})

Agora vá para a aba **Runs** do job. Em até ~1 minuto uma execução aparece **sozinha**, com o gatilho **File arrival**.

Quando terminar, rode a célula abaixo. A bronze ingeriu **só o arquivo novo**, e a silver ganhou os empréstimos novos sem duplicar os antigos.

In [0]:
%sql
SELECT _source_file, COUNT(*) AS linhas, MAX(_ingested_at) AS ingerido_em
FROM growdev.bronze.loans
GROUP BY _source_file
ORDER BY ingerido_em

### Parte 4: conferindo o histórico
Aba **Runs** do job: a coluna **Trigger** mostra quem disparou cada execução (*Manual*, *Scheduled*, *File arrival*).


## Checklist da aula
- [ ] Configurei um agendamento com fuso horário
- [ ] Pausei e retomei o agendamento
- [ ] Configurei o gatilho por chegada de arquivo
- [ ] Simulei a chegada de um arquivo e vi o job disparar sozinho
- [ ] Conferi no histórico quem disparou cada execução